# Stage 3: re-export the adapted model (Qwen3-1.7B + 10,000 trained Urdu tokens)

Converts the adapted model from notebook 07 to GGUF, **proves tokenizer parity** (Hugging Face and llama.cpp give identical ids on every FLORES sentence and a stress set; the export stops otherwise), builds Q4_K_M and Q8_0 with the Urdu importance matrix, writes the Ollama `Modelfile` and vLLM config, then runs `zubaan export check`:

- **Generation samples** on f16, Q8_0 and Q4_K_M: Urdu completion, Urdu chat (thinking off), Urdu chat with thinking allowed (to see the `<think>` block), and an English control. Greedy decoding.
- **Quantization cost**: KL divergence of Q8_0 and Q4_K_M against f16 on FLORES Urdu and FLORES English (same tokenizer, so per-token numbers are comparable between the files).

Settings before running:

- Accelerator: **None** (CPU). The GPU is not used: this notebook costs no GPU hours.
- Internet: **on**. No secret is needed.
- Input: **Add Input > Notebooks**: the output of `07_adapt_main_qwen3` (it holds `adapted/model/`). Do not add notebook 03's output; this notebook needs exactly one trained model.

Needs about 12 GB RAM and 20 GB disk. Expected time about 2 hours on 4 CPUs: conversion 5 min, parity 10 to 15, importance matrix 15 to 25, quantizing 5, generation samples 20 to 30, quantization cost 25 to 35.

**Before the first run:** the commit in the install cell must be pushed to GitHub. Use *Save Version > Save & Run All (Commit)*; finished files are skipped if you run it again.

**Download afterwards:** `export/export_summary.json`, `export/export_check.json`, `export/{NAME}-Q4_K_M.gguf` (about 1.2 GB, for the local Ollama check) and `export/Modelfile`.

llama.cpp release `b12477`, as in Stage 2.

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "6f427a1"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,export] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.export.check  # noqa: F401  (export check is in this commit)
import zubaan.export.parity  # noqa: F401
import zubaan.extend.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import shutil

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: the machine has what the run needs. No GPU is needed.
with open("/proc/meminfo") as f:
    mem = {line.split(":")[0]: int(line.split()[1]) for line in f}
ram_gb = mem["MemAvailable"] / 1e6
disk_gb = shutil.disk_usage("/kaggle/working").free / 1e9
n_cpu = os.cpu_count()
print(f"RAM available {ram_gb:.1f} GB, disk free {disk_gb:.1f} GB, CPUs {n_cpu}")
assert ram_gb >= 12, "need about 12 GB free RAM (1.7B in bfloat16 plus the GGUF conversion)"
assert disk_gb >= 20, "need about 20 GB free disk"
print("OK: machine is big enough")

In [ ]:
BASE_MODEL = "Qwen/Qwen3-1.7B"   # tokenizer reference for the pre-tokenizer and fingerprint checks
NAME = "qwen3-1.7b-urdu-adapted"

## 1. Find the adapted model

In [ ]:
import json
from pathlib import Path

# The adapted model is the output of notebook 07 (Add Input > Notebooks). Its zubaan_extension.json
# says trained: true, which tells it apart from the untrained extended model of notebook 03.
markers = sorted(Path("/kaggle/input").rglob("zubaan_extension.json"))
trained = [m.parent for m in markers if json.loads(m.read_text()).get("trained")]
print("trained model folders:", [str(f) for f in trained])
assert len(trained) == 1, f"need exactly one trained model folder, found {len(trained)}: add notebook 07's output"
MODEL_DIR = trained[0]
for f in ("tokenizer.json", "tokenizer_config.json", "config.json"):
    assert (MODEL_DIR / f).exists(), f"no {f} in {MODEL_DIR}"
assert (MODEL_DIR / "model.safetensors").exists() or list(MODEL_DIR.glob("model-*.safetensors")), "no weights"
meta = json.loads((MODEL_DIR / "zubaan_extension.json").read_text())
print(json.dumps(meta, indent=1)[:2000])
has_template = "chat_template" in (MODEL_DIR / "tokenizer_config.json").read_text() or (MODEL_DIR / "chat_template.jinja").exists()
assert has_template, "no chat template: Ollama and vLLM chat would fail"
print("OK: adapted model found:", MODEL_DIR)

## 2. llama.cpp

In [ ]:
import subprocess
from pathlib import Path

TAG = "b12477"
TOOLS = Path("/tmp/tools")  # outside /kaggle/working so it is not saved as output
LLAMA_DIR = TOOLS / f"llama.cpp-{TAG}"
LLAMA_DIR.mkdir(parents=True, exist_ok=True)
base = "https://github.com/ggml-org/llama.cpp"
# Source tree: the converter script and gguf-py.
!curl -sL {base}/archive/refs/tags/{TAG}.tar.gz | tar xz --strip-components=1 -C {LLAMA_DIR}
# Prebuilt CPU tools (llama-quantize, llama-imatrix, llama-tokenize, llama-completion).
BIN_ROOT = TOOLS / "bin"
BIN_ROOT.mkdir(exist_ok=True)
!curl -sL {base}/releases/download/{TAG}/llama-{TAG}-bin-ubuntu-x64.tar.gz | tar xz -C {BIN_ROOT}
found = sorted(BIN_ROOT.rglob("llama-quantize"))
assert found, "llama-quantize not found in the downloaded archive"
BIN_DIR = found[0].parent
print("llama.cpp tools in", BIN_DIR)

In [ ]:
# CHECK 3: the prebuilt tools run on this machine. If one does not (a library is missing),
# build llama.cpp from the source tree instead: about 10 minutes on 4 CPUs.
def tools_work(bin_dir):
    for tool in ("llama-quantize", "llama-imatrix", "llama-tokenize", "llama-completion", "llama-perplexity"):
        r = subprocess.run([str(bin_dir / tool), "--help"], capture_output=True, text=True)
        out = (r.stdout or "") + (r.stderr or "")
        # llama-quantize prints its usage and exits with 1, so exit codes 0 and 1 both count as "ran".
        if r.returncode not in (0, 1) or not out.strip() or "shared libraries" in out:
            print(tool, "failed:", r.returncode, out[-300:])
            return False
    return True


if not tools_work(BIN_DIR):
    print("prebuilt tools do not run here; building from source")
    !cmake -S {LLAMA_DIR} -B {LLAMA_DIR}/build -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF > /dev/null
    !cmake --build {LLAMA_DIR}/build --config Release -j {n_cpu} --target llama-quantize llama-imatrix llama-tokenize llama-completion llama-perplexity
    BIN_DIR = LLAMA_DIR / "build" / "bin"
    assert tools_work(BIN_DIR), "built tools do not run either"
print("OK: llama.cpp tools run, bin dir:", BIN_DIR)

## 3. Data and config

FLORES-200 (parity and quantization-cost texts) and Urdu Wikipedia (importance matrix).

In [ ]:
import os

import yaml

WORK = Path("/kaggle/working")
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "export": {
        "model_dir": str(MODEL_DIR),
        "base_model": BASE_MODEL,
        "output_dir": str(WORK / "export"),
        "name": NAME,
        "llama_cpp_dir": str(LLAMA_DIR),
        "llama_cpp_bin": str(BIN_DIR),
        "quants": ["Q4_K_M", "Q8_0"],
        # Importance matrix on Urdu training text (same as Stage 2). The quantization cost below is
        # measured on FLORES, which the model never trained on.
        "calibration": "data/urdu/urdu_train.txt",
        "calibration_chars": 200000,
        "imatrix_chunks": 40,
        "threads": os.cpu_count(),
        "parity_urdu": "data/flores_urd.txt",
        "parity_english": "data/flores_eng.txt",
        "parity_workers": os.cpu_count(),
        "max_model_len": 4096,
        "check_urdu": "data/flores_urd.txt",
        "check_english": "data/flores_eng.txt",
        "check_chunks": 20,
        "check_ctx": 512,
        "check_n_predict": 80,
        # The f16 logits file is about 3 GB per language: keep it out of /kaggle/working.
        "check_scratch": "/tmp/zubaan_check",
    },
}
Path("export.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("export.yaml").read_text())

In [ ]:
!zubaan data flores --out data
!zubaan data urdu --config export.yaml

## 4. Export with the parity check

In [ ]:
!zubaan export gguf --config export.yaml

In [ ]:
!zubaan export ollama --config export.yaml
!zubaan export vllm --config export.yaml

In [ ]:
summary = json.loads(Path("/kaggle/working/export/export_summary.json").read_text())
g = summary["gguf"]
p = g["parity"]
print("llama.cpp", g["llama_cpp_tag"], "| pre-tokenizer", g["pre_tokenizer"], "| converter patched:", g["converter_patched"])
print(f"parity: {p['n_texts'] - p['n_mismatch']}/{p['n_texts']} identical, {p['n_nfc_only']} differ only because the text is not NFC, {p['n_real_mismatch']} real mismatches")
assert p["passed"] and p["n_real_mismatch"] == 0, "tokenizer parity failed: see export_summary.json"
assert {"Q4_K_M", "Q8_0"} <= set(g["quantized"]), "a quantized file is missing"
# Stage 2 saw 2,041 of 2,045 identical (4 NFC-only). Training does not touch the tokenizer, so the
# same counts are expected here.
print("OK: GGUF export of the adapted model passed tokenizer parity")

## 5. Generation samples and quantization cost

In [ ]:
!zubaan export check --config export.yaml

In [ ]:
check = json.loads(Path("/kaggle/working/export/export_check.json").read_text())
for model, samples in check["samples"].items():
    print("=" * 30, model)
    for s in samples:
        share = s["answer_urdu_letter_share"]
        speed = s["tokens_per_s"]
        print(f"--- {s['prompt']} | think block: {s['has_think_block']} | Urdu letters: {share if share is None else round(share, 2)} | {speed} tok/s")
        print(s["input"])
        print(s["text"])
print()
for lang, res in check["quant_cost"].items():
    print(lang, "f16 PPL", round(res["f16_ppl"]["value"], 3))
    for q, k in res["quants"].items():
        print(f"  {q}: mean KLD {k['mean_kld']['value']:.4f} ± {k['mean_kld']['uncertainty']:.4f}, "
              f"same top token {k['same_top_p_percent']['value']:.1f}%, PPL ratio {k['ppl_ratio']['value']:.4f}")

In [ ]:
# CHECK 4 (success check 5): the Q4_K_M answers Urdu prompts in Urdu script. This only catches drift
# into another script; whether the Urdu reads well is for a person to judge from the text above.
q4 = {s["prompt"]: s for s in check["samples"]["Q4_K_M"]}
for name in ("urdu_completion_capital", "urdu_completion_lahore", "urdu_chat_capital", "urdu_chat_sentences"):
    share = q4[name]["answer_urdu_letter_share"]
    print(name, share)
    assert share is not None and share > 0.8, f"{name}: the Q4_K_M answer is not mostly Urdu script"
print("OK: Q4_K_M answers Urdu prompts in Urdu script. Now read the samples.")

## 6. Clean up and list the output

In [ ]:
# Kaggle keeps /kaggle/working as output (limit 20 GB). The f16 GGUF was needed for the parity and
# quantization checks; it is not a deliverable, so remove it.
WORK = Path("/kaggle/working")
for f in (WORK / "export").glob("*-F16.gguf"):
    print("removing", f.name, f"{f.stat().st_size / 1e9:.1f} GB")
    f.unlink()
total = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.stat().st_size > 5e6:
        total += f.stat().st_size
        print(f"{f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(WORK)}")
print(f"total of files over 5 MB: {total / 1e9:.1f} GB")
assert total < 19e9, "output is close to Kaggle's 20 GB limit"